In [1]:
# ─────────────────────────────────────────────
# STEP 1: Training corpus
# This is the raw text the model will learn from.
# The model can only predict words that appear here.
# ─────────────────────────────────────────────
faqs = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program's curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don't have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don't have to wait for a month to end.
What if I don't like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [2]:
# ─────────────────────────────────────────────
# STEP 2: Imports
# ─────────────────────────────────────────────
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [3]:
# STEP 3: Tokenizer
# Tokenizer converts words to integer indices.

tokenizer = Tokenizer()

In [4]:
tokenizer.fit_on_texts([faqs])

In [5]:
# Total number of unique words in the corpus.
print('Total unique words:', len(tokenizer.word_index))

Total unique words: 282


In [6]:
# We add +1 because index 0 is reserved for padding and is not
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary size (including padding index 0):', vocab_size)

Vocabulary size (including padding index 0): 283


In [7]:
# STEP 4
# For each sentence we create all possible prefix sequences.
input_sequences = []

for sentence in faqs.split('\n'):
    # Convert each word in the sentence to its integer token
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    # Generate all prefix n-grams of length 2 to len(sentence)
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i + 1])

In [8]:
max_len = max([len(x) for x in input_sequences])
print('Max sequence length:', max_len)

Max sequence length: 57


In [9]:
# STEP 5: Pad sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
print('Padded sequences shape:', padded_input_sequences.shape)

Padded sequences shape: (863, 57)


In [10]:
# STEP 6: Split into features (X) and labels (y)

X = padded_input_sequences[:, :-1]  # All columns except last → input context
y = padded_input_sequences[:, -1]   # Last column only → target next word

print('X shape (samples, context_length):', X.shape)
print('y shape (samples,):', y.shape)

X shape (samples, context_length): (863, 56)
y shape (samples,): (863,)


In [11]:
# STEP 7: One-hot encode labels
y = to_categorical(y, num_classes=vocab_size)
print('y shape after one-hot encoding:', y.shape)

y shape after one-hot encoding: (863, 283)


In [12]:

# STEP 8: Build the LSTM model
model = Sequential()
model.add(Embedding(vocab_size, 100, input_length=max_len - 1))
model.add(LSTM(150, return_sequences=True))
model.add(LSTM(150))
model.add(Dense(vocab_size, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
# STEP 9: Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [14]:
# STEP 10: Train the model

model.fit(X, y, epochs=100, validation_split=0.2)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.0768 - loss: 5.4195 - val_accuracy: 0.0462 - val_loss: 5.4984
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0855 - loss: 5.0274 - val_accuracy: 0.0462 - val_loss: 5.7383
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.0855 - loss: 4.9441 - val_accuracy: 0.0462 - val_loss: 5.8365
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0855 - loss: 4.9254 - val_accuracy: 0.0462 - val_loss: 5.9574
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0855 - loss: 4.9115 - val_accuracy: 0.0462 - val_loss: 6.0429
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0855 - loss: 4.9117 - val_accuracy: 0.0462 - val_loss: 6.0754
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0855 - loss: 4.9051 - val_accuracy: 0.0462 - val_loss: 6.1197
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0855 - loss: 4.8913 - val_accuracy: 0.

In [15]:
model.save('next_word_lstm.h5')
print('Model saved as next_word_lstm.h5')

Model saved as next_word_lstm.h5


In [16]:
# STEP 11: Build a reverse word index for fast lookup

reverse_word_index = {index: word for word, index in tokenizer.word_index.items()}
print('Reverse index built. Example:', list(reverse_word_index.items())[:5])

Reverse index built. Example: [(1, 'the'), (2, 'you'), (3, 'i'), (4, 'to'), (5, 'a')]


In [17]:
# STEP 13: Predict the next 10 words

text = "what is the fee"

for i in range(10):
    # Convert seed text to integer token sequence
    token_text = tokenizer.texts_to_sequences([text])[0]

    # Pad to match the input length the model was trained on
    padded_token_text = pad_sequences([token_text], maxlen=max_len - 1, padding='pre')

    # Get the index of the word with the highest predicted probability
    pos = np.argmax(model.predict(padded_token_text))

    # Look up the word for that index using the reverse dictionary (O(1))
    predicted_word = reverse_word_index.get(pos, '')

    # Append predicted word to the growing text
    text = text + ' ' + predicted_word
    print(text)
    time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step
what is the fee for
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
what is the fee for data
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is the fee for data science
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
what is the fee for data science mentorship
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
what is the fee for data science mentorship program
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
what is the fee for data science mentorship program dsmp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
what is the fee for data science mentorship program dsmp 2023
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
what is the fee for data science mentorship program dsmp 2023 2023
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
what is the fee for data science mentorship program dsmp 2023 2023 you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
what is the fee for data science mentorship program dsmp 2023 2023 you you


In [18]:
#inspect the full vocabulary (word → index mapping)
tokenizer.word_index

{'the': 1,
 'you': 2,
 'i': 3,
 'to': 4,
 'a': 5,
 'of': 6,
 'is': 7,
 'have': 8,
 'will': 9,
 'can': 10,
 'what': 11,
 'course': 12,
 'program': 13,
 'in': 14,
 'for': 15,
 'all': 16,
 'sessions': 17,
 'on': 18,
 'be': 19,
 'and': 20,
 'this': 21,
 'if': 22,
 'am': 23,
 'pay': 24,
 'payment': 25,
 'make': 26,
 'we': 27,
 'do': 28,
 'subscription': 29,
 'where': 30,
 'rs': 31,
 'so': 32,
 'campusx': 33,
 'session': 34,
 'our': 35,
 'paid': 36,
 'join': 37,
 'able': 38,
 'your': 39,
 'website': 40,
 'placement': 41,
 'fee': 42,
 'data': 43,
 'monthly': 44,
 'month': 45,
 'not': 46,
 'get': 47,
 'yes': 48,
 'once': 49,
 'past': 50,
 'feb': 51,
 'assistance': 52,
 'science': 53,
 '7': 54,
 '5600': 55,
 'are': 56,
 'watch': 57,
 'google': 58,
 'by': 59,
 'com': 60,
 'mail': 61,
 'from': 62,
 'contact': 63,
 'us': 64,
 'at': 65,
 'or': 66,
 'doubt': 67,
 'mentorship': 68,
 'payments': 69,
 '799': 70,
 'total': 71,
 'duration': 72,
 'months': 73,
 'learning': 74,
 'case': 75,
 'here': 76,
 '